# Cointegration Analysis — NIFTY 50 Stocks

This notebook identifies pairs of stocks that exhibit a long-term
equilibrium relationship using the Engle–Granger cointegration test.

Cointegrated pairs are potential candidates for statistical arbitrage
strategies such as pairs trading.


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import coint

## Load Cleaned Price Data

We load the cleaned closing price dataset generated in the previous
data preprocessing step.


In [ ]:
close_prices = pd.read_csv(
    "../data/processed/nifty50_close_prices.csv",
    index_col="Date",
    parse_dates=True
)

close_prices.head()

In [ ]:
def find_cointegrated_pairs(data):

    # number of assets
    n = data.shape[1]

    # matrices to store test statistics and p-values
    score_matrix = np.zeros((n, n))
    pvalue_matrix = np.ones((n, n))

    keys = data.columns
    pairs = []

    for i in range(n):
        for j in range(i+1, n):

            # select two price series
            S1 = data[keys[i]]
            S2 = data[keys[j]]

            # perform Engle–Granger cointegration test
            score, pvalue, _ = coint(S1, S2)

            score_matrix[i, j] = score
            pvalue_matrix[i, j] = pvalue

            # store pair if statistically significant
            if pvalue < 0.01:
                pairs.append((keys[i], keys[j]))


    return score_matrix, pvalue_matrix, pairs

## Log Transformation of Prices

We transform prices using logarithms to stabilize variance and
improve statistical properties for time-series testing.


In [ ]:
log_prices=np.log(close_prices)

## Running Cointegration Tests

We apply the Engle–Granger cointegration test to every pair
of stocks to detect statistically significant relationships.


In [ ]:
scores, pvalues, pairs = find_cointegrated_pairs(log_prices)

print("Total pairs tested:", len(log_prices.columns)*(len(log_prices.columns)-1)//2)

## Ranking Cointegrated Pairs

Pairs are ranked based on p-values from the cointegration test.
Lower p-values indicate stronger statistical evidence of cointegration.


In [ ]:
pvalue_df = pd.DataFrame(
    pvalues,
    index=log_prices.columns,
    columns=log_prices.columns
)

sorted_pairs = (
    pvalue_df
    .where(np.triu(np.ones(pvalue_df.shape), k=1).astype(bool))
    .stack()
    .sort_values()
)

print("Best cointegrated pairs:")
print(sorted_pairs.head(10))

## Extract Significant Cointegrated Pairs

Pairs with p-values below 0.05 are selected as statistically
significant candidates for pairs trading strategies.


In [ ]:
cointegrated_pairs = sorted_pairs[sorted_pairs < 0.05].sort_values()
print("Total cointegrated pairs:", len(cointegrated_pairs))
print("Significance threshold: p-value < 0.05")


cointegrated_pairs.to_csv("../data/processed/cointegrated_pairs.csv")

print("Cointegrated pairs saved successfully")


## Visualization of Cointegration Strength

The heatmap below shows p-values from the cointegration test
between all stock pairs.

Darker colors represent stronger cointegration relationships.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

sns.heatmap(
    pvalue_df,
    cmap="viridis",
    vmax=0.05
)

plt.title("Cointegration P-Value Matrix")
plt.savefig("../reports/cointegration_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


## Output

This notebook produces:

• Cointegration test results for all NIFTY 50 stock pairs  
• Ranked list of statistically significant cointegrated pairs  
• Cointegration p-value heatmap visualization  

The final dataset of significant cointegrated pairs is saved to:

data/processed/cointegrated_pairs.csv
